# Task 3: End-to-End ABSA with PhoBERT
## Unified Model: Extraction + Classification in ONE model

**Evaluation** (thong nhat voi NB03, NB05, phobert-crf-absa.ipynb):
- Span-Level Exact Match F1
- Sentence-Level Multi-Label F1 (Micro/Macro/Weighted + per-label)


## 1. Setup


In [ ]:
import subprocess, sys, os

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'underthesea', 'pytorch-crf', 'transformers', 'py_vncorenlp'])

IS_KAGGLE = os.path.exists('/kaggle/input')
if IS_KAGGLE:
    DATA_DIR = '/kaggle/input/datasets/danghoang1302/uit-visd4sa'
    SAVE_DIR = '/kaggle/working/results/e2e'
    SRC_INPUT = '/kaggle/input/datasets/danghoang1302/absa-src'
    os.system(f'cp -r {SRC_INPUT}/src /kaggle/working/src')
    sys.path.insert(0, '/kaggle/working')
    print(f"KAGGLE | Data: {DATA_DIR}")
else:
    sys.path.insert(0, os.path.abspath(".."))
    DATA_DIR = os.path.join("..", "..", "data")
    SAVE_DIR = os.path.join("..", "..", "results", "e2e")
    print(f"LOCAL mode")

os.makedirs(SAVE_DIR, exist_ok=True)
for fn in ['train.jsonl', 'dev.jsonl', 'test.jsonl']:
    assert os.path.exists(os.path.join(DATA_DIR, fn)), f"MISSING: {fn}"
print("All data files OK!")


## 2. Imports & Load Data


In [ ]:
import torch, torch.nn as nn, numpy as np, pandas as pd
from torch.utils.data import DataLoader
from IPython.display import display

from src.utils.preprocess import load_raw_data
from src.e2e.e2e_dataset import E2EDataset, BIO_TAGS, TAG2ID, NUM_TAGS, LABEL_NAMES
from src.e2e.e2e_model import E2EPhoBertCRF
from src.utils.engine import train_e2e_model, predict_e2e
from src.utils.metrics import (bio_tags_to_spans, evaluate_spans_f1, token_accuracy,
                               bio_to_sentence_labels, evaluate_multilabel)
from src.utils.visualization import plot_training_curves, plot_model_comparison

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42); np.random.seed(42)
print(f"Device: {device}")

train_items = load_raw_data(os.path.join(DATA_DIR, "train.jsonl"))
dev_items = load_raw_data(os.path.join(DATA_DIR, "dev.jsonl"))
test_items = load_raw_data(os.path.join(DATA_DIR, "test.jsonl"))
print(f"Train: {len(train_items)} | Dev: {len(dev_items)} | Test: {len(test_items)}")


## 3. E2E Dataset & Train


In [ ]:
MAX_LEN = 256
BATCH_SIZE = 16

print("Tokenizing with PhoBERT...")
train_ds = E2EDataset(train_items, max_len=MAX_LEN)
dev_ds = E2EDataset(dev_items, max_len=MAX_LEN)
test_ds = E2EDataset(test_items, max_len=MAX_LEN)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_ds, BATCH_SIZE)
test_loader = DataLoader(test_ds, BATCH_SIZE)
print(f"Ready! Train: {len(train_ds)} | Dev: {len(dev_ds)} | Test: {len(test_ds)}")


In [ ]:
LR_OPTIONS = [2e-5, 5e-5]
all_models, all_histories, all_results = {}, {}, []

for lr in LR_OPTIONS:
    name = f"PhoBERT-CRF_lr{lr}"
    model = E2EPhoBertCRF(num_unified_tags=NUM_TAGS, dropout=0.3).to(device)
    model, history = train_e2e_model(
        model, train_loader, dev_loader, device,
        lr=lr, epochs=15, patience=5, model_name=name)
    all_models[name] = model
    all_histories[name] = history


## 4. Evaluate on Test Set


In [ ]:
best_f1_global, best_model_key, best_mt, best_test_res = 0, "", None, None

for name, model in all_models.items():
    test_res = predict_e2e(model, test_loader, device)
    pred_spans = [bio_tags_to_spans(pt, BIO_TAGS, l) for pt, l in zip(test_res['pred_tags'], test_res['lengths'])]
    true_spans = [bio_tags_to_spans(tt, BIO_TAGS, l) for tt, l in zip(test_res['true_tags'], test_res['lengths'])]
    span_f1 = evaluate_spans_f1(pred_spans, true_spans)

    pred_sent, _ = bio_to_sentence_labels(test_res['pred_tags'], test_res['lengths'], BIO_TAGS, LABEL_NAMES)
    true_sent, _ = bio_to_sentence_labels(test_res['true_tags'], test_res['lengths'], BIO_TAGS, LABEL_NAMES)
    mt = evaluate_multilabel(true_sent, pred_sent, LABEL_NAMES)

    all_results.append({"Model": name, "Span_F1": round(span_f1['f1'], 4),
        "Micro_F1": round(mt['micro']['f1'], 4), "Macro_F1": round(mt['macro']['f1'], 4)})
    print(f"{name} | Span F1: {span_f1['f1']:.4f} | Micro F1: {mt['micro']['f1']:.4f}")

    if span_f1['f1'] > best_f1_global:
        best_f1_global = span_f1['f1']
        best_model_key = name
        best_mt = mt
        best_test_res = test_res
        best_pred_spans = pred_spans
        best_true_spans = true_spans

e2e_df = pd.DataFrame(all_results)
display(e2e_df)

print(f"\n  BEST: {best_model_key}")
print(f"  {'Label':<25} {'P':>7} {'R':>7} {'F1':>7} {'Sup':>6}")
print(f"  {'-'*55}")
for ln in LABEL_NAMES:
    m = best_mt[ln]
    print(f"  {ln:<25} {m['precision']:>7.4f} {m['recall']:>7.4f} {m['f1']:>7.4f} {m['support']:>6d}")


## 5. Demo: Test Predictions


In [ ]:
import random
random.seed(42)
n_samples = min(10, len(test_items))
sample_indices = random.sample(range(len(test_items)), n_samples)

print(f"{'='*70}")
print(f"  E2E PhoBERT-CRF DEMO: {n_samples} cau test")
print(f"{'='*70}")

demo_tp, demo_fp, demo_fn = 0, 0, 0
for idx in sample_indices:
    text = test_items[idx]['text']
    words = text.split()
    true_s = best_true_spans[idx]
    pred_s = best_pred_spans[idx]
    true_set, pred_set = set(true_s), set(pred_s)
    demo_tp += len(true_set & pred_set)
    demo_fp += len(pred_set - true_set)
    demo_fn += len(true_set - pred_set)

    print(f"\n{'─'*70}")
    print(f"  [{idx}] {text[:100]}{'...' if len(text)>100 else ''}")
    print(f"  TRUE ({len(true_s)}):")
    for label, s, e in true_s:
        aspect = ' '.join(words[s:e]) if e <= len(words) else '?'
        match = 'OK' if (label, s, e) in pred_set else 'MISSED'
        print(f"    {label:<30} [{s}:{e}] \"{aspect}\"  {match}")
    print(f"  PRED ({len(pred_s)}):")
    for label, s, e in pred_s:
        aspect = ' '.join(words[s:e]) if e <= len(words) else '?'
        match = 'OK' if (label, s, e) in true_set else 'WRONG'
        print(f"    {label:<30} [{s}:{e}] \"{aspect}\"  {match}")
    if not pred_s:
        print(f"    (khong co prediction)")

print(f"\n{'='*70}")
print(f"  Demo total: TP={demo_tp}, FP={demo_fp}, FN={demo_fn}")


## 6. Save


In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
e2e_df.to_csv(os.path.join(SAVE_DIR, "e2e_results.csv"), index=False)
best_model = all_models[best_model_key]
torch.save(best_model.state_dict(), os.path.join(SAVE_DIR, "best_e2e_phobert.pt"))
print(f"Best E2E: {best_model_key} (Span F1={best_f1_global:.4f}) -> Saved!")
